In [1]:
import numpy as np

In [2]:
sentence = "What are the symptoms of diabetes"
tokens = sentence.split()
vocab = {word: idx for idx, word in enumerate(tokens)}
token_ids = [vocab[t] for t in tokens]
print("Tokens:   ", tokens)
print("Token IDs:", token_ids)

Tokens:    ['What', 'are', 'the', 'symptoms', 'of', 'diabetes']
Token IDs: [0, 1, 2, 3, 4, 5]


In [3]:
np.random.seed(42)
d_model = 8          
n_tokens = len(tokens)
 
E = np.random.randn(n_tokens, d_model)  
X = E[token_ids]  

In [4]:
def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            PE[pos, i]   = np.sin(pos / (10000 ** (i / d_model)))
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(pos / (10000 ** (i / d_model)))
    return PE
 
PE = positional_encoding(n_tokens, d_model)
X = X + PE   
print(f"\nInput matrix X (with positional encoding): shape {X.shape}")


Input matrix X (with positional encoding): shape (6, 8)


In [5]:
d_k = 8   
 
W_Q = np.random.randn(d_model, d_k)
W_K = np.random.randn(d_model, d_k)
W_V = np.random.randn(d_model, d_k)

In [6]:
Q = X @ W_Q   
K = X @ W_K   
V = X @ W_V   
 
print(f"\nQ shape: {Q.shape}, K shape: {K.shape}, V shape: {V.shape}")


Q shape: (6, 8), K shape: (6, 8), V shape: (6, 8)


In [7]:
scores_raw = Q @ K.T / np.sqrt(d_k)   
print(f"\nRaw attention scores (Q·Kᵀ / √d_k): shape {scores_raw.shape}")
print(np.round(scores_raw, 3))


Raw attention scores (Q·Kᵀ / √d_k): shape (6, 6)
[[-3.1081e+01  9.4910e+00 -5.3400e-01 -7.6680e+00  1.3210e+01 -8.3260e+00]
 [ 6.9710e+00 -1.6990e+00  6.4000e-01  3.4890e+00 -4.3350e+00  2.8970e+00]
 [-8.1000e-02  1.1090e+00 -1.8950e+00 -1.6700e-01 -3.0430e+00  3.9800e-01]
 [ 2.5000e-01  3.9930e+00  2.6050e+00 -4.4200e-01 -9.6170e+00 -1.3860e+00]
 [ 1.9233e+01 -1.2984e+01  5.9330e+00 -2.5060e+00 -1.7539e+01 -4.2590e+00]
 [ 9.0000e-03  1.3570e+00  8.4300e-01 -3.7800e-01 -5.5850e+00 -2.1000e-02]]


In [8]:
def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True)) 
    return e_x / e_x.sum(axis=axis, keepdims=True)
 
attention_weights = softmax(scores_raw, axis=-1)   
print(f"\nAttention weights (after softmax): shape {attention_weights.shape}")
print(np.round(attention_weights, 3))
print("\nEach row sums to 1.0:", np.allclose(attention_weights.sum(axis=1), 1.0))


Attention weights (after softmax): shape (6, 6)
[[0.    0.024 0.    0.    0.976 0.   ]
 [0.953 0.    0.002 0.029 0.    0.016]
 [0.142 0.467 0.023 0.131 0.007 0.23 ]
 [0.018 0.775 0.194 0.009 0.    0.004]
 [1.    0.    0.    0.    0.    0.   ]
 [0.114 0.437 0.261 0.077 0.    0.11 ]]

Each row sums to 1.0: True


In [9]:
attention_output = attention_weights @ V   
print(f"\nAttention output (Attention_weights · V): shape {attention_output.shape}")
print(np.round(attention_output, 3))


Attention output (Attention_weights · V): shape (6, 8)
[[ 2.062 -4.91   2.368  0.644 -3.141  4.84   0.912 -0.802]
 [-0.503  1.126  6.455  3.312 -7.499 -4.055 -1.222  1.923]
 [ 0.559 -1.466  3.853  2.546 -4.263 -2.074 -0.266  2.142]
 [ 1.153  2.025  1.805  3.026 -1.219 -1.434 -0.267  2.729]
 [-0.522  1.43   6.506  3.412 -7.627 -4.178 -1.303  1.969]
 [ 0.525  1.127  2.466  2.291 -1.692 -1.58  -0.212  1.874]]


In [10]:
print("\n── Attention Weight Matrix (rows=query token, cols=key token) ──")
print(f"{'':>12}", end="")
for t in tokens:
    print(f"{t:>12}", end="")
print()
for i, t in enumerate(tokens):
    print(f"{t:>12}", end="")
    for j in range(n_tokens):
        print(f"{attention_weights[i,j]:>12.3f}", end="")
    print()


── Attention Weight Matrix (rows=query token, cols=key token) ──
                    What         are         the    symptoms          of    diabetes
        What       0.000       0.024       0.000       0.000       0.976       0.000
         are       0.953       0.000       0.002       0.029       0.000       0.016
         the       0.142       0.467       0.023       0.131       0.007       0.230
    symptoms       0.018       0.775       0.194       0.009       0.000       0.004
          of       1.000       0.000       0.000       0.000       0.000       0.000
    diabetes       0.114       0.437       0.261       0.077       0.000       0.110


In [11]:
print("\n── For each token, the key it attends to most ──")
for i, qt in enumerate(tokens):
    best_j = np.argmax(attention_weights[i])
    print(f"  '{qt}' → '{tokens[best_j]}'  (weight={attention_weights[i, best_j]:.3f})")


── For each token, the key it attends to most ──
  'What' → 'of'  (weight=0.976)
  'are' → 'What'  (weight=0.953)
  'the' → 'are'  (weight=0.467)
  'symptoms' → 'are'  (weight=0.775)
  'of' → 'What'  (weight=1.000)
  'diabetes' → 'are'  (weight=0.437)
